In [1]:
import os
# Force CPU backend on Apple Silicon to avoid Metal issues
os.environ['JAX_PLATFORMS'] = 'cpu'

import matplotlib.pyplot as plt

# Disable LaTeX rendering in matplotlib
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "sans-serif"

from jax import random
from jax import numpy as jnp
from sbijax import plot_loss_profile

from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.simulator import PatchForagingDDM_JAX, create_prior
from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.snle.snle_inference_jax import train_snle, infer_parameters_snle
from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.snle.snle_utils_jax import plot_real_synth_hist, extract_samples

In [ ]:
# --- Setup ---
num_window_sites = 100
n_simulations = 750_000
simulator = PatchForagingDDM_JAX(max_sites_per_window=num_window_sites)

# Get prior bounds for JAX simulator
prior_fn = create_prior()
rng_key = random.PRNGKey(0)

# --- Train SNLE ---
print("\n1. Training SNLE model...")
snle, snle_params, losses, rng_key, y_mean, y_std = train_snle(
    simulator, 
    prior_fn,
    mode='multi', 
    n_simulations=n_simulations,
    num_layers = 12,
    n_iter=1000,                        # Ensure full 1000 iterations possible
    n_early_stopping_patience=50,       # More patience for convergence
    batch_size=512,                     # Can adjust based on memory
    percentage_data_as_validation_set = .1, #.1 is default try .2
    rng_key=rng_key
)

# Disable LaTeX rendering in matplotlib
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "sans-serif"

_, axes = plt.subplots(figsize=(6, 3))
plot_loss_profile(losses, axes)
plt.show()

In [ ]:
losses

In [ ]:
# Check if the effects are actually detectable in the raw data
print("\n=== EFFECT SIZE ANALYSIS ===")

from jax import random
import jax
from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.simulator import PatchForagingDDM_JAX, create_prior

# Test if new stats improve parameter-statistic correlations

rng_key = random.PRNGKey(123)
n_test = 1000

# --- Setup ---
num_window_sites = 100
simulator = PatchForagingDDM_JAX(max_sites_per_window=num_window_sites)

# Get prior bounds for JAX simulator
prior_fn = create_prior()


# Test extreme parameter values
test_params = [
    jnp.array([0.5, 0.1, 0.1, 0.05]),  # minimal bumps, low noise
    jnp.array([0.5, 0.9, 0.1, 0.05]),  # high reward bump
    jnp.array([0.5, 0.1, 0.9, 0.05]),  # high failure bump
    jnp.array([0.5, 0.5, 0.5, 0.4]),   # high noise
]

labels = ["minimal", "high_reward", "high_failure", "high_noise"]

rng_key = random.PRNGKey(999)
for theta, label in zip(test_params, labels):
    rng_key, subkey = random.split(rng_key)
    window_data, stats = simulator.simulate_one_window(theta, subkey)
    
    patch_times = window_data[:, 0]
    rewards = window_data[:, 1]
    stops = window_data[:, 2]
    
    # Calculate actual effect sizes
    prev_rewards = jnp.roll(rewards, 1).at[0].set(0)
    valid = stops > 0
    after_reward = patch_times[(prev_rewards > 0) & valid & (jnp.arange(len(patch_times)) > 0)]
    after_failure = patch_times[(prev_rewards == 0) & valid & (jnp.arange(len(patch_times)) > 0)]
    
    print(f"\n{label}: theta={theta}")
    print(f"  Mean time after reward:  {jnp.mean(after_reward):.3f} ± {jnp.std(after_reward):.3f}")
    print(f"  Mean time after failure: {jnp.mean(after_failure):.3f} ± {jnp.std(after_failure):.3f}")
    print(f"  Difference: {jnp.mean(after_failure) - jnp.mean(after_reward):.3f}")
    print(f"  Effect size (Cohen's d): {(jnp.mean(after_failure) - jnp.mean(after_reward)) / jnp.sqrt((jnp.var(after_reward) + jnp.var(after_failure))/2):.3f}")


=== EFFECT SIZE ANALYSIS ===


NameError: name 'simulator' is not defined

In [ ]:
# --- Simulate observed data ---
print("\n2. Simulating observed data...")
rng_key, subkey = random.split(rng_key)
true_theta = prior_fn().sample(seed=subkey)['theta']
rng_key, subkey = random.split(rng_key)
_, observed_stats = simulator.simulate_one_window(true_theta, subkey)
print(f"   True theta: {true_theta}")
print(f"   Observed stats: {observed_stats}")

In [ ]:
# --- Run inference ---
print("\n3. Testing inference...")
rng_key, subkey = random.split(rng_key)
posterior_samples, diagnostics = infer_parameters_snle(
snle,
snle_params,
observed_stats, 
y_mean, y_std,
num_samples=100_000,
num_warmup=50,
num_chains=2,
rng_key=subkey
)

In [ ]:
# --- Plot posterior distributions ---
param_names = ["drift_rate", "reward_bump", "failure_bump", "noise_std"]
param_labels = [
    "drift_rate: evidence accumulation rate",
    "reward_bump: evidence boost from receiving reward",
    "failure_bump: evidence boost from not receiving reward",
    "noise_std: std of noise in evidence accumulation"
]

fig, axes = plt.subplots(1, 4, figsize=(10, 2))
axes = axes.flatten()

for i in range(4):

    # Compute histogram
    counts, bins, _ =axes[i].hist(posterior_samples[:, i], bins=30, color='dodgerblue', edgecolor=None, alpha=0.7)

    # Posterior mode (bin center with max count)
    mode_index = jnp.argmax(counts)
    posterior_mode = (bins[mode_index] + bins[mode_index + 1]) / 2

    axes[i].axvline(true_theta[i], color='orangered', linestyle='--', label="true value")
    axes[i].axvline(posterior_mode, color='k', linestyle='--', label='MAP estimate')

    axes[i].set_xlabel(param_names[i])
    axes[i].set_ylabel("Frequency")

axes[i].legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

def pairplot(posterior_samples, true_params=None, param_names=None, figsize_per_param=2.5, grid_points=100):
    """
    Lower-triangle corner plot with:
    - 2D filled KDEs (off-diagonal)
    - 1D KDEs (diagonal)
    - Red 'X' for true parameters
    """
    if isinstance(posterior_samples, jnp.ndarray):
        posterior_samples = np.array(posterior_samples)
    
    n_params = posterior_samples.shape[1]
    if param_names is None:
        param_names = [f"param{i}" for i in range(n_params)]
    
    fig, axes = plt.subplots(n_params, n_params, figsize=(figsize_per_param*n_params, figsize_per_param*n_params))
    
    for i in range(n_params):
        for j in range(n_params):
            ax = axes[i, j]
            
            # Only fill lower triangle
            if i < j:
                ax.axis('off')
                continue
            
            # Diagonal: 1D KDE
            if i == j:
                data = posterior_samples[:, i]
                kde = gaussian_kde(data)
                x_grid = np.linspace(data.min(), data.max(), grid_points)
                ax.fill_between(x_grid, kde(x_grid), color="skyblue")
                
                if true_params is not None:
                    ax.axvline(true_params[i], color='red', linestyle='--', lw=1)
            
            # Off-diagonal: 2D KDE
            else:
                x = posterior_samples[:, j]
                y = posterior_samples[:, i]
                xy = np.vstack([x, y])
                kde = gaussian_kde(xy)
                x_grid = np.linspace(x.min(), x.max(), grid_points)
                y_grid = np.linspace(y.min(), y.max(), grid_points)
                X, Y = np.meshgrid(x_grid, y_grid)
                Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
                ax.contourf(X, Y, Z, levels=20, cmap="Blues")
                
                if true_params is not None:
                    ax.scatter(true_params[j], true_params[i], c='red', s=50, marker='X', label='True')
            
            # Only label left and bottom axes
            if i < n_params - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel(param_names[j])
            if j > 0:
                ax.set_yticklabels([])
            else:
                ax.set_ylabel(param_names[i])
    
    # Add a legend in the top-left subplot
    handles = []
    if true_params is not None:
        handles.append(plt.Line2D([0], [0], marker='X', color='w', markerfacecolor='red', markersize=8, label='True'))
    axes[0, 1].legend(handles=handles, loc='upper left')
    
    plt.tight_layout()
    plt.show()

In [ ]:
pairplot(posterior_samples, true_theta, param_names, figsize_per_param=2.0)

In [ ]:
# Generate multiple patches from posterior samples to compare distributions
print("\nGenerating patches from posterior samples for comparison...")

num_window_sites = 500

#--- Simulate 'real' data from simulator---
print("\n2. Simulating observed data...")
real_data = []
for i_site in range(num_window_sites):
    rng_key, subkey = random.split(rng_key)
    _, data = simulator.simulate_one_window(true_theta, subkey)
    real_data.append(data)
real_data = np.array(real_data)

# # 2. Generate "synthetic" data from simulator using posterior-sampled parameters
# Get observed summary stats
rng_key, subkey = random.split(rng_key)
_, observed_stats = simulator.simulate_one_window(true_theta, subkey)
obs_norm = (observed_stats - y_mean) / y_std

# Sample posterior θ using ONE observation
rng_key, sub = random.split(rng_key)
inference_results, diagnostics = snle.sample_posterior(
    sub, snle_params,
    observable=obs_norm,
    n_samples=500,
    n_chains=1,
    n_warmup=200,
)

posterior_ds = extract_samples(inference_results)
theta_samples = posterior_ds["theta"].values.reshape(-1, 4)  # shape (500,4)

# Simulate synthetic patches for comparison
synthetic = []
for theta in theta_samples:
    rng_key, subkey = random.split(rng_key)
    _, stats = simulator.simulate_one_window(theta, subkey)
    synthetic.append(stats)

synthetic = jnp.array(synthetic)

# 4. Plot
plot_real_synth_hist(real_data, synthetic)



In [ ]:
# 1. Check if parameters actually affect summary statistics
print("\n=== PARAMETER SENSITIVITY TEST ===")
test_params = [
    jnp.array([0.05, 0.5, 0.5, 0.1]),  # very low drift
    jnp.array([0.5, 0.5, 0.5, 0.1]),   # medium drift
    jnp.array([0.95, 0.5, 0.5, 0.1]),  # high drift
]

rng_key = random.PRNGKey(42)
for i, theta in enumerate(test_params):
    rng_key, subkey = random.split(rng_key)
    window_data, stats = simulator.simulate_one_window(theta, subkey)
    
    n_patches = jnp.sum(window_data[:, 2])
    print(f"\nTheta {i}: drift={theta[0]:.2f}")
    print(f"  n_patches: {n_patches}")
    print(f"  mean_time: {stats[1]:.3f}")
    print(f"  time_after_reward: {stats[7]:.3f}")
    print(f"  time_after_failure: {stats[8]:.3f}")

# 2. Visualize parameter-statistic relationships from training data
import matplotlib.pyplot as plt

# Get training data
rng_key, data_key = random.split(rng_key)
data, _ = snle.simulate_data(data_key, n_simulations=5000)
theta_train = data['theta']['theta']
y_train = data['y']

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
key_stat_indices = [1, 7, 8, 11, 12]  # mean_time, after_reward, after_failure, early, late
key_stat_names = ['mean_time', 'after_reward', 'after_failure', 'early_mean', 'late_mean']

for i in range(4):  # parameters
    for j, (stat_idx, stat_name) in enumerate(zip(key_stat_indices, key_stat_names)):
        ax = axes[i, j] if j < 4 else axes[i, 0]
        if j < len(key_stat_indices):
            ax.scatter(theta_train[:, i], y_train[:, stat_idx], alpha=0.1, s=1)
            ax.set_xlabel(param_names[i])
            ax.set_ylabel(stat_name)
            
            # Compute correlation
            corr = jnp.corrcoef(theta_train[:, i], y_train[:, stat_idx])[0, 1]
            ax.set_title(f'r={corr:.3f}')

plt.tight_layout()
plt.show()

# 3. Check distribution of patches per simulation
rng_key, data_key = random.split(rng_key)
n_patches_list = []
for i in range(1000):
    rng_key, subkey = random.split(rng_key)
    theta = prior_fn().sample(seed=subkey)['theta']
    window_data, _ = simulator.simulate_one_window(theta, subkey)
    n_patches = jnp.sum(window_data[:, 2])
    n_patches_list.append(n_patches)

plt.figure(figsize=(8, 4))
plt.hist(n_patches_list, bins=50)
plt.xlabel('Number of patches per simulation')
plt.ylabel('Count')
plt.title('Distribution of patch counts')
plt.show()

print(f"Mean patches: {jnp.mean(jnp.array(n_patches_list)):.1f}")
print(f"Min patches: {jnp.min(jnp.array(n_patches_list)):.0f}")
print(f"Max patches: {jnp.max(jnp.array(n_patches_list)):.0f}")

In [ ]:
# Visualize parameter-statistic relationships from your ACTUAL training data
import matplotlib.pyplot as plt

# Extract training data that was actually used
rng_key = random.PRNGKey(123)
rng_key, data_key = random.split(rng_key)
data, _ = snle.simulate_data(data_key, n_simulations=5000)
theta_train = data['theta']['theta']
y_train = data['y']

param_names = ["drift_rate", "reward_bump", "failure_bump", "noise_std"]

# Check correlations for KEY statistics
key_stat_indices = [1, 7, 8, 11, 12, 25]  # mean_time, after_reward, after_failure, early, late, n_patches
key_stat_names = ['mean_time', 'after_reward', 'after_failure', 'early_mean', 'late_mean', 'n_patches']

fig, axes = plt.subplots(4, 6, figsize=(18, 10))

for i in range(4):  # parameters
    for j, (stat_idx, stat_name) in enumerate(zip(key_stat_indices, key_stat_names)):
        ax = axes[i, j]
        ax.scatter(theta_train[:, i], y_train[:, stat_idx], alpha=0.1, s=1)
        ax.set_xlabel(param_names[i] if i == 3 else '')
        ax.set_ylabel(stat_name if i == 0 else '')
        
        # Compute correlation
        corr = jnp.corrcoef(theta_train[:, i], y_train[:, stat_idx])[0, 1]
        ax.set_title(f'r={corr:.2f}')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print correlation matrix
print("\n=== CORRELATION MATRIX ===")
print("Parameter vs Key Statistics:")
print(f"{'Param':<15} {'mean_time':>10} {'after_rew':>10} {'after_fail':>10} {'n_patches':>10}")
print("-" * 60)
for i, pname in enumerate(param_names):
    corrs = [jnp.corrcoef(theta_train[:, i], y_train[:, idx])[0, 1] 
             for idx in [1, 7, 8, 25]]
    print(f"{pname:<15} {corrs[0]:>10.3f} {corrs[1]:>10.3f} {corrs[2]:>10.3f} {corrs[3]:>10.3f}")

In [ ]:
# Test if the network learned ANYTHING
print("\n=== FORWARD MODEL TEST ===")

# Generate test data
rng_key = random.PRNGKey(999)
test_thetas = jnp.array([
    [0.2, 0.5, 0.5, 0.1],
    [0.8, 0.5, 0.5, 0.1],
    [0.5, 0.2, 0.8, 0.1],
    [0.5, 0.8, 0.2, 0.1],
])

for theta in test_thetas:
    # Simulate true data
    rng_key, subkey = random.split(rng_key)
    _, true_stats = simulator.simulate_one_window(theta, subkey)
    
    # Get posterior samples
    rng_key, subkey = random.split(rng_key)
    posterior_samples, _ = infer_parameters_snle(
        snle, snle_params, true_stats, y_mean, y_std,
        num_samples=1000, num_warmup=100, num_chains=2, rng_key=subkey
    )
    
    post_mean = posterior_samples.mean(axis=0)
    post_std = posterior_samples.std(axis=0)
    
    print(f"\nTrue:  {theta}")
    print(f"Infer: {post_mean} ± {post_std}")